# PAS label summary (Which object × class)

Same logic as `summarize_pas_labels.py`: scans `data/` for `*_PAS_m_label_V*.csv` and aggregates counts. (This is **not** for Qwen/SAM3/YOLO exports — those live under `outputs/images/` from `qwen_sam3_image_pipeline.ipynb`.)

**Google Drive (Google Colab)**

1. Upload this notebook to Colab (or open from GitHub).
2. Run the cell below to **mount Drive** and set `DATA_ROOT` to the folder that contains your `data` tree (the folder that has `*_PAS_m_label_*.csv` files inside subfolders).
3. Optionally set `OUTPUT_CSV` to something like `/content/drive/MyDrive/pas_summary.csv` to save results on Drive.

**Local (Cursor / Jupyter)**

Leave `MOUNT_GOOGLE_DRIVE = False`, keep `DATA_ROOT` pointing at `auto-labeler/data`, run all cells.


In [8]:
# --- Google Colab: mount Drive (skip when running locally) ---
MOUNT_GOOGLE_DRIVE = False  # set True in Colab

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

# Root folder that CONTAINS the PAS data tree (must include subfolders with CSVs)
# Colab example after sync:
# DATA_ROOT = Path("/content/drive/MyDrive/your_folder/auto-labeler/data")
from pathlib import Path

DATA_ROOT = Path("data").resolve()
if not DATA_ROOT.is_dir():
    alt = Path.cwd() / "auto-labeler" / "data"
    alt = "/Users/ral3ply/Google Drive/My Drive/auto-labeler"
    if alt.is_dir():
        DATA_ROOT = alt.resolve()

# Latest V* per stem only (set False to count every CSV file, including V1+V2)
EVERY_FILE = False

# Optional: save summary CSV (local path or Drive path in Colab)
OUTPUT_CSV: Path | None = None  # e.g. Path("/content/drive/MyDrive/pas_summary.csv")

print("DATA_ROOT:", DATA_ROOT)

DATA_ROOT: /Users/ral3ply/Documents/GitHub/visual-search/auto-labeler/data


In [2]:
import csv
import sys
from pathlib import Path

# Resolve auto-labeler directory (this notebook + summarize_pas_labels.py)
_here = Path.cwd()
if (_here / "summarize_pas_labels.py").is_file():
    AUTO_LABELER = _here
elif (_here / "auto-labeler" / "summarize_pas_labels.py").is_file():
    AUTO_LABELER = _here / "auto-labeler"
else:
    AUTO_LABELER = _here

if str(AUTO_LABELER) not in sys.path:
    sys.path.insert(0, str(AUTO_LABELER))

from summarize_pas_labels import aggregate_pas_labels

print("Using module from:", AUTO_LABELER)

Using module from: /Users/ral3ply/Documents/GitHub/visual-search/auto-labeler


In [3]:
summary = aggregate_pas_labels(DATA_ROOT, every_file=EVERY_FILE)

print(
    f"Files on disk: {len(summary.all_csv_paths)} CSVs | "
    f"Used: {len(summary.chosen_csv_paths)} | "
    f"Rows: {summary.total_rows} | "
    f"Non-empty files: {summary.files_with_rows}"
)

Files on disk: 28 CSVs | Used: 11 | Rows: 24 | Non-empty files: 11


In [4]:
try:
    import pandas as pd
except ImportError:
    pd = None

if pd is not None:
    rows = [
        {"which_object": o, "class": c, "count": n}
        for (o, c), n in sorted(summary.pair_counts.items(), key=lambda x: (-x[1], x[0][0], x[0][1]))
    ]
    df_pairs = pd.DataFrame(rows)
    display(df_pairs)

    display(pd.DataFrame([{"class": c, "count": n} for c, n in summary.class_counts.most_common()]))
    display(pd.DataFrame([{"which_object": o, "count": n} for o, n in summary.object_counts.most_common()]))
else:
    print("Install pandas for tables: pip install pandas")
    for (o, c), n in summary.pair_counts.most_common():
        print(o, c, n)

,which_object,class,count
0,empty,0,14
1,wall,3,8
2,car,3,1
3,hedge,3,1


,class,count
0,0,14
1,3,10


,which_object,count
0,empty,14
1,wall,8
2,hedge,1
3,car,1


In [5]:
if OUTPUT_CSV is not None:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["which_object", "class", "count"])
        for (obj, cls), n in sorted(summary.pair_counts.items()):
            w.writerow([obj, cls, n])
    print("Wrote", OUTPUT_CSV)
else:
    print("Set OUTPUT_CSV in the config cell to save a CSV (e.g. on Google Drive).")

Set OUTPUT_CSV in the config cell to save a CSV (e.g. on Google Drive).
